TAYYAB MAQSOOD (2023_AG-9687)

KASHUF AFTAB (2023-AG-9562)

BSCS M3

CS-506 BIG DATA

### **Movie Ratings Analysis using PySpark**

### **Install PySpark**

In [15]:
!pip install pyspark

### **Start Spark Session**

In [16]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Movie Rating Analysis") \
    .getOrCreate()

### **Load Movie Dataset**

In [17]:
movies = spark.read.csv("movies.csv", header=True, inferSchema=True)

movies.show(5)

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows


### **Load Ratings Dataset**

In [18]:
ratings = spark.read.csv("ratings.csv", header=True, inferSchema=True)

ratings.show(5)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows


### **Join Both Datasets**

In [19]:
movie_ratings = ratings.join(movies, on="movieId", how="inner")

movie_ratings.show(5)

+-------+------+------+---------+--------------------+--------------------+
|movieId|userId|rating|timestamp|               title|              genres|
+-------+------+------+---------+--------------------+--------------------+
|      1|     1|   4.0|964982703|    Toy Story (1995)|Adventure|Animati...|
|      3|     1|   4.0|964981247|Grumpier Old Men ...|      Comedy|Romance|
|      6|     1|   4.0|964982224|         Heat (1995)|Action|Crime|Thri...|
|     47|     1|   5.0|964983815|Seven (a.k.a. Se7...|    Mystery|Thriller|
|     50|     1|   5.0|964982931|Usual Suspects, T...|Crime|Mystery|Thr...|
+-------+------+------+---------+--------------------+--------------------+
only showing top 5 rows


### **Find Average Rating Per Movie**

In [20]:
from pyspark.sql.functions import avg

average_ratings = movie_ratings.groupBy("title") \
    .agg(avg("rating").alias("average_rating"))

average_ratings.show(10)

+--------------------+------------------+
|               title|    average_rating|
+--------------------+------------------+
|       Psycho (1960)| 4.036144578313253|
|Men in Black (a.k...| 3.487878787878788|
|Gulliver's Travel...|               3.0|
|Heavenly Creature...|3.9285714285714284|
|    Elizabeth (1998)|3.6739130434782608|
|Before Night Fall...|               4.3|
|O Brother, Where ...|3.8085106382978724|
|Snow White and th...| 3.616883116883117|
| Three Wishes (1995)|               3.0|
|When We Were King...|               3.9|
+--------------------+------------------+
only showing top 10 rows


### **Count Number of Ratings**

In [21]:
from pyspark.sql.functions import count

rating_counts = movie_ratings.groupBy("title") \
    .agg(count("rating").alias("number_of_ratings"))

rating_counts.show(10)

+--------------------+-----------------+
|               title|number_of_ratings|
+--------------------+-----------------+
|       Psycho (1960)|               83|
|Men in Black (a.k...|              165|
|Gulliver's Travel...|                3|
|Heavenly Creature...|               21|
|    Elizabeth (1998)|               23|
|Before Night Fall...|                5|
|O Brother, Where ...|               94|
|Snow White and th...|               77|
| Three Wishes (1995)|                1|
|When We Were King...|               10|
+--------------------+-----------------+
only showing top 10 rows


### **Filter Top Rated Movies**

In [22]:
movie_stats = movie_ratings.groupBy("title") \
    .agg(
        avg("rating").alias("average_rating"),
        count("rating").alias("number_of_ratings")
    )

### **Get Top 10 Movies**

In [23]:
top_movies = movie_stats.orderBy(
    ["average_rating", "number_of_ratings"],
    ascending=False
)

top_movies.show(10)

+--------------------+--------------+-----------------+
|               title|average_rating|number_of_ratings|
+--------------------+--------------+-----------------+
|Heidi Fleiss: Hol...|           5.0|                2|
|     Lamerica (1994)|           5.0|                2|
| Lesson Faust (1994)|           5.0|                2|
| Belle époque (1992)|           5.0|                2|
|Come and See (Idi...|           5.0|                2|
|Enter the Void (2...|           5.0|                2|
|Jonah Who Will Be...|           5.0|                2|
|Bill Hicks: Revel...|           5.0|                1|
|      Villain (1971)|           5.0|                1|
|Shogun Assassin (...|           5.0|                1|
+--------------------+--------------+-----------------+
only showing top 10 rows


### **Saving Output File**

In [29]:
top_10_movies = top_movies.limit(10)

top_10_movies.toPandas().to_csv("top_10_movies.csv", index=False)

### **Downloading Output File**

In [31]:
from google.colab import files

files.download("top_10_movies.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>